# Cài đặt thư viện và Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers datasets scikit-learn

import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, set_seed
from sklearn.metrics import accuracy_score, f1_score

# Đọc dữ liệu và Ánh xạ nhãn

In [ ]:
# 1. Đọc dữ liệu

train_df = pd.read_csv('/content/train.csv')
val_df = pd.read_csv('/content/val.csv')
test_df = pd.read_csv('/content/test.csv')

# 2. Map nhãn chuẩn: fake=0, real=1
label_map = {'fake': 0, 'real': 1}
for df in [train_df, val_df, test_df]:
    df['label'] = df['label'].map(label_map)

# 3. Chuyển đổi sang format Dataset của Hugging Face
train_dataset = Dataset.from_pandas(train_df[['tweet', 'label']])
val_dataset = Dataset.from_pandas(val_df[['tweet', 'label']])
test_dataset = Dataset.from_pandas(test_df[['tweet', 'label']])

# Tokenizer(Tiền xử lý văn bản)

In [ ]:
# Tải tokenizer của BERT-base
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding và cắt gọn văn bản (max_length có thể tinh chỉnh, ví dụ 128)
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=128)

# Áp dụng tokenizer lên toàn bộ tập dữ liệu
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

# Thiết lập đánh giá và Vòng lặp huấn luyện với 3 Seed

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Trả về các metric trên thang 0-1
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

# Yêu cầu 3 seed
seeds = [42, 123, 2024]

for seed in seeds:
    print(f"\n{'='*40}")
    print(f"BẮT ĐẦU HUẤN LUYỆN VỚI SEED: {seed}")
    print(f"{'='*40}")

    # Cố định seed cho khả năng tái tạo (reproducibility)
    set_seed(seed)

    # Khởi tạo mô hình MỚI cho mỗi lần lặp seed
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

    # Tạo thư mục xuất kết quả riêng rẽ cho từng seed
    output_dir = f'./bert_base_seed_{seed}'

    # Cấu hình Training
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,                     # Số vòng lặp huấn luyện, có thể tăng lên 4 hoặc 5 tùy bạn
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",            # Đánh giá sau mỗi epoch
        save_strategy="epoch",                  # Lưu checkpoint sau mỗi epoch
        metric_for_best_model="f1_weighted",    # TIÊU CHÍ QUAN TRỌNG: Chọn checkpoint tốt nhất bằng f1_weighted
        load_best_model_at_end=True,            # Tự động nạp lại checkpoint tốt nhất vào cuối quá trình train
        seed=seed
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics
    )

    # Bắt đầu train
    trainer.train()

    # Đánh giá lại trên tập Validation để ghi log
    val_results = trainer.evaluate()
    print(f"\nKết quả trên tập Validation (Seed {seed}):")
    print(val_results)


BẮT ĐẦU HUẤN LUYỆN VỚI SEED: 42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,No log,0.093291,0.972430,0.972429
2,0.165223,0.109465,0.977103,0.977098
3,0.039966,0.123548,0.977570,0.977565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.044583,0.089852,3,0.980841,0.980838



Kết quả trên tập Validation (Seed 2024):
{'eval_loss': 0.0898517295718193, 'eval_accuracy': 0.9808411214953271, 'eval_f1_weighted': 0.9808379801806025}


# Error Analysis (Phân tích lỗi)

In [6]:
import numpy as np

print("Đang tiến hành phân tích lỗi trên tập Validation...")
# Dự đoán đồng loạt bằng trainer (tối ưu hơn pipeline rất nhiều)
predictions, labels, _ = trainer.predict(val_tokenized)
preds = np.argmax(predictions, axis=1)

# Lọc ra các vị trí dự đoán sai
error_indices = np.where(preds != labels)[0]
print(f"Tổng số câu dự đoán sai: {len(error_indices)}\n")

# In ra phân tích 10 mẫu đầu tiên
for idx, i in enumerate(error_indices[:10]):
    tweet = val_dataset[int(i)]['tweet']
    true_label = val_dataset[int(i)]['label']
    pred_label = preds[i]

    true_text = "Real (1)" if true_label == 1 else "Fake (0)"
    pred_text = "Real (1)" if pred_label == 1 else "Fake (0)"

    print(f"--- Mẫu sai số {idx + 1} ---")
    print(f"Thực tế: {true_text} | Mô hình đoán: {pred_text}")
    print(f"Nội dung: {tweet}\n")

Đang tiến hành phân tích lỗi trên tập Validation...


Tổng số câu dự đoán sai: 41

--- Mẫu sai số 1 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: There is no evidence that children have died because of a COVID-19 vaccine. No vaccine currently in development has been approved for widespread public use. https://t.co/9ecvMR8SAf

--- Mẫu sai số 2 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: The Canadian government has not enacted a law that criminalizes COVID-19 restrictions or prevents enforcement of those guidelines. https://t.co/c7IST1A1ip

--- Mẫu sai số 3 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: IACC is to Autism as "X" is to COVID https://t.co/447cYfVImc

--- Mẫu sai số 4 ---
Thực tế: Real (1) | Mô hình đoán: Fake (0)
Nội dung: A reminder about maintaining good hygiene: wash your hands often cough or sneeze into your elbow and stay at home if you’re unwell.

--- Mẫu sai số 5 ---
Thực tế: Fake (0) | Mô hình đoán: Real (1)
Nội dung: An article from April announcing a new study “connects” the flu vacc